In [0]:
spark

## Data Ingestion

In [0]:
df = spark.table("workspace.default.fraud_dataset")

## Understanding the Data

In [0]:
display(df.limit(10))

step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
1,TRANSFER,181.0,C1305486145,181.0,0.0,C553264065,0.0,0.0,1,0
1,CASH_OUT,181.0,C840083671,181.0,0.0,C38997010,21182.0,0.0,1,0
1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0
1,PAYMENT,7817.71,C90045638,53860.0,46042.29,M573487274,0.0,0.0,0,0
1,PAYMENT,7107.77,C154988899,183195.0,176087.23,M408069119,0.0,0.0,0,0
1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.0,0.0,0,0
1,PAYMENT,4024.36,C1265012928,2671.0,0.0,M1176932104,0.0,0.0,0,0
1,DEBIT,5337.77,C712410124,41720.0,36382.23,C195600860,41898.0,40348.79,0,0


In [0]:
df.count()

6362620

In [0]:
df.printSchema()

root
 |-- step: long (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: double (nullable = true)
 |-- newbalanceOrig: double (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: double (nullable = true)
 |-- newbalanceDest: double (nullable = true)
 |-- isFraud: long (nullable = true)
 |-- isFlaggedFraud: long (nullable = true)



In [0]:
df.columns

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud']

In [0]:
display(df.describe())

summary,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
count,6362620,6362620,6362620,6362620,6362620,6362620,6362620,6362620,6362620,6362620,6362620
mean,243.39724563151657,null,179861.90354913095,null,833883.1040744836,855113.6685785847,null,1100701.6665196405,1224996.398201917,0.001290820448180152,2.51468734577894E-6
stddev,142.33197104912983,null,603858.2314629357,null,2888242.673037557,2924048.502954267,null,3399180.1129944758,3674128.9421196356,0.03590479680160415,0.0015857747057365491
min,1,CASH_IN,0.0,C1000000639,0.0,0.0,C1000004082,0.0,0.0,0,0
max,743,TRANSFER,9.244551664E7,C999999784,5.958504037E7,4.958504037E7,M999999784,3.5601588935E8,3.5617927892E8,1,1


## Exploratory Data Analysis

### Fraud Counts

In [0]:
df.groupBy("isFraud").count().show()

+-------+-------+
|isFraud|  count|
+-------+-------+
|      1|   8213|
|      0|6354407|
+-------+-------+



In [0]:
df.groupBy("isFlaggedFraud").count().show()

+--------------+-------+
|isFlaggedFraud|  count|
+--------------+-------+
|             0|6362604|
|             1|     16|
+--------------+-------+



In [0]:
from pyspark.sql.functions import mean

df.agg(mean("isFraud")).show()

+--------------------+
|        avg(isFraud)|
+--------------------+
|0.001290820448180152|
+--------------------+



### Transaction Types

In [0]:
tran_types = df.groupBy("type").count().orderBy("count", ascending=False)
display(tran_types)

type,count
CASH_OUT,2237500
PAYMENT,2151495
CASH_IN,1399284
TRANSFER,532909
DEBIT,41432


In [0]:
tran_types.plot.bar(x="type", y="count", title="Distribution of Transaction Types", template="plotly_white")

### Fraud Rate by Transaction Types

In [0]:
from pyspark.sql.functions import avg

fraud_by_type = df.groupBy("type").agg(avg("isFraud")).orderBy("avg(isFraud)", ascending=False)
display(fraud_by_type)

type,avg(isFraud)
TRANSFER,0.007687991758442811
CASH_OUT,0.0018395530726256983
PAYMENT,0.0
DEBIT,0.0
CASH_IN,0.0


In [0]:
fraud_by_type.plot.bar(x="type", y="avg(isFraud)", title="Fraud Rate by Transaction Types", template="plotly_white", \
    labels={"avg(isFraud)": "fraud_rate"}, color_discrete_sequence=["#5F7D8E"])

### Amount Statistics

In [0]:
df.describe("amount").show(truncate=False)

+-------+------------------+
|summary|amount            |
+-------+------------------+
|count  |6362620           |
|mean   |179861.90354913095|
|stddev |603858.2314629357 |
|min    |0.0               |
|max    |9.244551664E7     |
+-------+------------------+



In [0]:
from pyspark.sql.functions import log1p

df_with_log = df.withColumn("log_amount", log1p("amount"))

In [0]:
df_with_log.plot.hist(column="log_amount", bins=100, kde=True, title="Distribution of Transaction Amount (Log)",\
    template="plotly_white")

In [0]:
from pyspark.sql.functions import col

df.filter(df["amount"] < 50000).plot.box(column="amount")

In [0]:
df.filter(df["amount"] < 50000).show(10)

+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|  amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+--------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|  42| PAYMENT| 5660.56| C618230452|          0.0|           0.0|M1352937915|           0.0|           0.0|      0|             0|
|  42| PAYMENT| 5899.22|  C10795828|      80252.0|      74352.78|M1903979378|           0.0|           0.0|      0|             0|
|  42|CASH_OUT|10599.52|C1924354067|      10908.0|        308.48|C2136594180|           0.0|      10599.52|      0|             0|
|  42|CASH_OUT|28961.98| C614966351|      10238.0|           0.0| C830447488|     124431.92|      153393.9|      0|             0|
|  42| PAYMENT| 1390.85| C332008018|       1069.0|           0.0|M2064774128|      

### Frauds Over Time

In [0]:
frauds_per_step = df.filter(df["isFraud"] == 1).groupBy("step").count().orderBy("step")

display(frauds_per_step.limit(24))

step,count
1,16
2,8
3,4
4,10
5,6
6,22
7,12
8,12
9,19
10,11


In [0]:
frauds_per_step.plot.line(x="step", y="count", title="Fraud Transactions per Step", template="plotly_white", \
    labels={"count": "number of frauds", "step": "step (time)"})

### Top Senders & Receivers

In [0]:
top_senders = df.groupBy("nameOrig").count().orderBy("count", ascending=False)

top_senders.show(10)

+-----------+-----+
|   nameOrig|count|
+-----------+-----+
| C400299098|    3|
|C1677795071|    3|
|C1784010646|    3|
|C1065307291|    3|
|C1530544995|    3|
| C363736674|    3|
|C1902386530|    3|
|C1999539787|    3|
|C2098525306|    3|
|C1832548028|    3|
+-----------+-----+
only showing top 10 rows


In [0]:
top_receivers = df.groupBy("nameDest").count().orderBy("count", ascending=False)

top_receivers.show(10)

+-----------+-----+
|   nameDest|count|
+-----------+-----+
|C1286084959|  113|
| C985934102|  109|
| C665576141|  105|
|C2083562754|  102|
|C1590550415|  101|
| C248609774|  101|
|C1789550256|   99|
| C451111351|   99|
|C1360767589|   98|
|C1023714065|   97|
+-----------+-----+
only showing top 10 rows


### Zero Balance after Transfer

In [0]:
zero_after_transfer = df.filter(\
    (df["oldbalanceOrg"] > 0) &\
    (df["newbalanceOrig"] == 0) &\
    (df["type"].isin("TRANSFER", "CASH_OUT")))

In [0]:
zero_after_transfer.show(10)

+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|step|    type|   amount|   nameOrig|oldbalanceOrg|newbalanceOrig|   nameDest|oldbalanceDest|newbalanceDest|isFraud|isFlaggedFraud|
+----+--------+---------+-----------+-------------+--------------+-----------+--------------+--------------+-------+--------------+
|  42|TRANSFER|599918.25|C1862740146|     158527.0|           0.0| C161137214|           0.0|     599918.25|      0|             0|
|  42|CASH_OUT| 28961.98| C614966351|      10238.0|           0.0| C830447488|     124431.92|      153393.9|      0|             0|
|  42|CASH_OUT|126473.08|C1653454081|      40838.0|           0.0| C153631072|           0.0|     126473.08|      0|             0|
|  42|CASH_OUT|411351.25| C714830066|        227.0|           0.0| C278046461|    5128761.76|    5540113.01|      0|             0|
|  42|CASH_OUT|446483.62|C1269342049|       6929.0|           0.0| C26125021

In [0]:
zero_after_transfer.count()

1188074

## Feature Engineering

In [0]:
df = df.withColumn("balanceDiffOrig", df["oldbalanceOrg"] - df["newbalanceOrig"])

In [0]:
df = df.withColumn("balanceDiffDest", df["oldbalanceDest"] - df["newbalanceDest"])

In [0]:
df.columns

['step',
 'type',
 'amount',
 'nameOrig',
 'oldbalanceOrg',
 'newbalanceOrig',
 'nameDest',
 'oldbalanceDest',
 'newbalanceDest',
 'isFraud',
 'isFlaggedFraud',
 'balanceDiffOrig',
 'balanceDiffDest']

## Generating a Modeling Dataset

In [0]:
df_model = df.drop("step", "nameOrig", "nameDest", "isFlaggedFraud")

In [0]:
df_model.show(10)

+--------+--------+-------------+--------------+--------------+--------------+-------+------------------+------------------+
|    type|  amount|oldbalanceOrg|newbalanceOrig|oldbalanceDest|newbalanceDest|isFraud|   balanceDiffOrig|   balanceDiffDest|
+--------+--------+-------------+--------------+--------------+--------------+-------+------------------+------------------+
| PAYMENT| 9839.64|     170136.0|     160296.36|           0.0|           0.0|      0| 9839.640000000014|               0.0|
| PAYMENT| 1864.28|      21249.0|      19384.72|           0.0|           0.0|      0|1864.2799999999988|               0.0|
|TRANSFER|   181.0|        181.0|           0.0|           0.0|           0.0|      1|             181.0|               0.0|
|CASH_OUT|   181.0|        181.0|           0.0|       21182.0|           0.0|      1|             181.0|           21182.0|
| PAYMENT|11668.14|      41554.0|      29885.86|           0.0|           0.0|      0|          11668.14|               0.0|


In [0]:
df_model.count()

6362620

In [0]:
import pandas as pd
import numpy as np

In [0]:
df_model = df_model.toPandas()

In [0]:
df_model.head()

,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest
0,PAYMENT,5660.56,0.0,0.00,0.00,0.00,0,0.00,0.00
1,CASH_IN,107551.46,300191.0,407742.46,223824.28,116272.81,0,-107551.46,107551.47
2,TRANSFER,599918.25,158527.0,0.00,0.00,599918.25,0,158527.00,-599918.25
3,PAYMENT,5899.22,80252.0,74352.78,0.00,0.00,0,5899.22,0.00
4,CASH_OUT,10599.52,10908.0,308.48,0.00,10599.52,0,10599.52,-10599.52


In [0]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 9 columns):
 #   Column           Dtype  
---  ------           -----  
 0   type             object 
 1   amount           float64
 2   oldbalanceOrg    float64
 3   newbalanceOrig   float64
 4   oldbalanceDest   float64
 5   newbalanceDest   float64
 6   isFraud          int64  
 7   balanceDiffOrig  float64
 8   balanceDiffDest  float64
dtypes: float64(7), int64(1), object(1)
memory usage: 436.9+ MB


In [0]:
df_model.describe()

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balanceDiffOrig,balanceDiffDest
count,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06,6.362620e+06
mean,1.798619e+05,8.338831e+05,8.551137e+05,1.100702e+06,1.224996e+06,1.290820e-03,-2.123056e+04,-1.242947e+05
std,6.038582e+05,2.888243e+06,2.924049e+06,3.399180e+06,3.674129e+06,3.590480e-02,1.466433e+05,8.129391e+05
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.915268e+06,-1.056878e+08
25%,1.338957e+04,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,-1.491054e+05
50%,7.487194e+04,1.420800e+04,0.000000e+00,1.327057e+05,2.146614e+05,0.000000e+00,0.000000e+00,0.000000e+00
75%,2.087215e+05,1.073152e+05,1.442584e+05,9.430367e+05,1.111909e+06,0.000000e+00,1.015044e+04,0.000000e+00
max,9.244552e+07,5.958504e+07,4.958504e+07,3.560159e+08,3.561793e+08,1.000000e+00,1.000000e+07,1.306083e+07


In [0]:
df_model["isFraud"].value_counts()


isFraud
0    6354407
1       8213
Name: count, dtype: int64